# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [3]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL (3+) TABLES HERE
"""} ]



In [7]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
-- Example 1: List all employee names and their salaries.
Q: Show the names of all employees and their salaries.
A: SELECT employees.name, salary.salary FROM employees JOIN salary ON employees.ID_usr = salary.ID_usr;

-- Example 2: Find employees who have a postgraduate degree.
Q: List employees who have a postgraduate educational level.
A: SELECT employees.name FROM employees JOIN studies ON employees.ID_usr = studies.ID_usr WHERE studies.educational_level = 'postgraduate';

-- Example 3: Show the average salary for each educational level.
Q: What is the average salary for each educational level?
A: SELECT studies.educational_level, AVG(salary.salary) as avg_salary FROM studies JOIN salary ON studies.ID_usr = salary.ID_usr GROUP BY studies.educational_level;
"""
})

In [8]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [12]:
#new
context_user = context.copy()
print(return_CCRMSQL("select employee name from employees (as e), join salaries to each employee, order by salary in desc and limit to 1", context_user))

```sql
SELECT e.name
FROM employees AS e
JOIN salary AS s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```


In [13]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("select employee name from employees (as e), join salaries to each employee, order by salary in desc and limit to 1", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of an employee from the "employees" table, joins it with the "salary" table based on the employee ID, orders the result by the salary in descending order, and limits the output to only one result.


In [15]:
#new
print(return_CCRMSQL("Write an SQL query to find the institution with the highest average salary. Use three tables: studies (alias st), employees (alias e), and salary (alias sa). The studies table contains ID_Usr and Institution. Join studies with employees on ID_Usr, and join employees with salary on ID_Usr. Calculate the average salary (AVG(sa.salary)) for each institution, and return the institution name and its average salary. Order the results by average salary in descending order, and limit the output to just one result.", context_user))

```sql
SELECT st.Institution, AVG(sa.salary) as avg_salary
FROM studies st
JOIN employees e ON st.ID_usr = e.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [17]:
#old
print(return_CCRMSQL("Write an SQL query that selects only the institution from a 'studies' table (aliased as s), joining it with a salary table (aliased as sa) on user ID, grouping by institution, ordering by the average salary in descending order, and limiting to 1 result.", old_context_user))

This is your SQL:
```sql
SELECT s.Institution
FROM studies s
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY AVG(sa.salary) DESC
LIMIT 1;
```

This SQL query selects the institution from the 'studies' table and joins it with the 'salary' table based on the user ID. It then groups the results by institution, calculates the average salary for each institution, orders the results by the average salary in descending order, and limits the output to only 1 result.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

# Report: Findings from Prompt Engineering for NL2SQL

## Overview

In this exercise, we explored different prompt engineering strategies for generating SQL queries from natural language using GPT models. We compared an "old prompt" (minimal context, JSON-like table definitions, and brief instructions) with a "new prompt" (SQL `CREATE TABLE` syntax, sample rows, and few-shot examples). Our goal was to assess which approaches led to more accurate, reliable, and interpretable SQL generation.

## Key Findings

### 1. Effectiveness of Prompt Structure

- **Few-shot examples** in the new prompt significantly improved SQL generation accuracy. By providing concrete question/answer pairs, the model better understood the expected SQL format and logic.
- Using **SQL `CREATE TABLE` syntax** and including **sample data** made the schema more explicit, reducing ambiguity and helping the model generate valid SQL that matched the database structure.

### 2. Variations That Did Not Work Well

- The **old prompt** sometimes led to hallucinations or incorrect SQL:
	- The model occasionally invented column names or misunderstood table relationships.
	- Without explicit examples, the model sometimes produced overly complex or inefficient queries.
- Prompts lacking **clear join conditions** or **ambiguous field names** resulted in SQL that did not match the intended schema.
- When the prompt was too verbose or included conflicting instructions, the model's output became inconsistent.

### 3. Observed Hallucinations and Errors

- In some cases, GPT generated SQL using non-existent columns or tables, especially when the schema was not clearly defined.
- Occasionally, the model ignored constraints (e.g., missing `GROUP BY` when aggregating) or produced invalid SQL syntax.
- The model sometimes defaulted to generic SQL patterns rather than adapting to the provided schema.

### 4. Lessons Learned

- **Explicitness matters:** The more precisely the schema and task are described, the better the results.
- **Few-shot learning is powerful:** Including relevant examples in the prompt guides the model toward the desired output format and logic.
- **Schema clarity:** Using standard SQL syntax for schema definition is more effective than informal or JSON-like descriptions.
- **Iterative refinement:** Testing and refining prompts is essential; small changes can have a large impact on output quality.

## Conclusion

Prompt engineering is crucial for reliable NL2SQL generation. Providing clear schema definitions, sample data, and few-shot examples leads to more accurate and robust SQL queries. Ambiguous or underspecified prompts increase the risk of hallucinations and errors. Iterative experimentation and careful prompt design are key to leveraging LLMs effectively for text-to-SQL tasks.